##### 코랩을 사용할 경우

In [ ]:
try:
    # Google Drive를 Colab(/content/drive)rmfja 에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/내 Mac/sec03
    %ls
except ImportError:
    pass

Drive already mounted at /google_drive; to attempt to forcibly remount, call drive.mount("/google_drive", force_remount=True).
/google_drive/Othercomputers/내 Mac/sec03
01_data_type_change.ipynb   03_device_change.ipynb
02_backward_gradient.ipynb  04_data_model_same_device.ipynb


##### 임포트

In [ ]:
import torch
import torch.nn as nn

##### 간단한 신경망 모델 정의

In [ ]:
# 간단한 신경망 모델 정의
# PyTorch에서 신경망을 만들 때는 nn.Module을 상속받아야 함
class SimpleModel(nn.Module):
    def __init__(self):
        # nn.Module의 초기화 메서드 호출
        super().__init__()

        # nn.Linear(입력_차원, 출력_차원): 완전 연결층(Fully Connected Layer)
        # 입력: 10개의 특성(features), 출력: 5개의 특성
        self.fc = nn.Linear(10, 5)

    def forward(self, x):
        # 순전파(forward pass) 정의
        # x: (batch_size, 10) 형태의 입력 텐서
        # 출력: (batch_size, 5) 형태의 텐서
        return self.fc(x)

##### 모델과 입력데이터가 다른 장치 메모리에 있을 경우

In [6]:
# 모델을 CPU에 생성
# - 모델의 파라미터(가중치, 편향)도 CPU 메모리에 저장됨
model_cpu = SimpleModel()

# model_cpu.parameters()는 모델의 모든 파라미터(가중치, 편향)를 반환하는 제너레이터 객체
# - 제너레이터는 인덱싱([0])을 사용할 수 없음
# - next(iterator 또는 generator): 반복 가능한 객체에서 다음 요소를 가져오는 함수
# - next(model_cpu.parameters()): 모델의 가중치와 편향을 레이어 순서대로 가져옴
print(f"모델 가중치가 존재하는 장치: {next(model_cpu.parameters()).device}")  # cpu 출력

# CPU에 데이터 생성 (기본값)
# torch.randn(1, 10): 평균 0, 표준편차 1인 정규분포에서 (1, 10) 크기의 랜덤 텐서 생성
data = torch.randn(1, 10)
print(f"입력 데이터 위치: {data_gpu.device}")  # cuda:0 출력 (GPU)

# GPU가 사용 가능하면 데이터를 GPU로 이동
if torch.cuda.is_available():
    # 입력 데이터를 GPU 메모리로 이동
    data_gpu = data.to("cuda")
    print(f"입력 데이터 위치: {data_gpu.device}")  # cuda:0 출력 (GPU)

    # GPU에 있는 데이터를 CPU에 있는 모델에 입력
    output = model_cpu(data_gpu)

모델 가중치가 존재하는 장치: cpu
입력 데이터 위치: cuda:0
입력 데이터 위치: cuda:0


RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

##### 모델과 입력 데이터가 모두 CPU 메모리에 있을 경우

In [7]:
# 모델을 CPU에 생성
model_cpu = SimpleModel()
print(f"모델 위치: {next(model_cpu.parameters()).device}")  # cpu

# 데이터를 CPU에 생성
data_cpu = torch.randn(1, 10)
print(f"입력 데이터 위치: {data_cpu.device}")  # cpu

# CPU에서 연산 수행: CPU에 있는 데이터를 CPU에 있는 모델에 입력
# 입력 데이터와 모델 가중치가 같은 CPU 장치에 있으므로 정상 작동
# CPU 연산은 GPU보다 느리지만, 작은 모델이나 추론 시에는 충분히 사용 가능
output = model_cpu(data_cpu)

# 출력 텐서도 자동으로 CPU에 생성됨
print(f"신경망 출력 위치: {output.device}")  # cpu
print(f"신경망 출력 형태: {output.shape}")  # torch.Size([1, 5])

모델 위치: cpu
입력 데이터 위치: cpu
신경망 출력 위치: cpu
신경망 출력 형태: torch.Size([1, 5])


##### 모델과 입력 데이터가 모두 GPU 메모리에 있을 경우

In [8]:
if torch.cuda.is_available():
    # 모델 생성 후 GPU로 이동
    # .to("cuda"): 모델의 모든 파라미터(가중치, 편향)를 GPU 메모리로 이동
    model_gpu = SimpleModel().to("cuda")
    print(f"모델 위치: {next(model_gpu.parameters()).device}")  # cuda:0

    # 데이터 생성 후 GPU로 이동
    # 방법 1: torch.randn(1, 10).to("cuda")
    # 방법 2: torch.randn(1, 10, device="cuda")
    data_gpu = torch.randn(1, 10).to("cuda")
    print(f"입력 데이터 위치: {data_gpu.device}")  # cuda:0

    # GPU에서 연산 수행: GPU에 있는 데이터를 GPU에 있는 모델에 입력
    # 입력 데이터와 모델 가중치가 같은 GPU 장치에 있으므로 정상 작동
    output = model_gpu(data_gpu)

    # 출력 텐서도 자동으로 GPU에 생성됨
    print(f"신경망 출력 위치: {output.device}")  # cuda:0
    print(f"신경망 출력 형태: {output.shape}")

모델 위치: cuda:0
입력 데이터 위치: cuda:0
신경망 출력 위치: cuda:0
신경망 출력 형태: torch.Size([1, 5])
